# ARCHIVED RESEARCH NOTE

**What this notebook works on:** Measures eight-step autonomous spectrogram rollout error accumulation when the model cannot see ground-truth intermediate future frames.

**Research question:** Music RQ2: Does coupled PDE–ODE forecasting reduce eight-step error accumulation and transient degradation relative to simpler and ablated models?

**Output:** On execution: music_multistep_metrics.csv, music_multistep_comparison.png, v2_complex_proposed_comparison.png and methodological figures; no executed cell outputs are stored.

**Evidence status:** Described outputs are generated only when the notebook runs; no positive result is inferred from the code alone. Existing experiment cells were preserved, and the active publication V2 notebook is not involved.

**Original Git blob:** `1e758be583b8091eaa196f694843f80e5dd83055`

See the companion research note and archive index in `notebooks/archive/`.

# Error Propagation in Autoregressive Music Spectrogram Forecasting with Coupled Neural PDE–ODE Dynamics

*Executable research protocol · Academic V2 · Google Colab*

## Abstract

One-step music spectrogram prediction can underestimate the difficulty of modeling temporal dynamics, because adjacent frames are strongly correlated and prediction errors may accumulate during autonomous rollout. This study examines the stability and short-horizon forecasting accuracy of a coupled neural PDE–ODE predictor when intermediate observations are withheld. A neural reaction–diffusion field represents local spectral interactions, while a low-dimensional ODE describes global state evolution. Following one-step estimation, the model generates eight consecutive future spectra by feeding its predictions back as inputs. Its horizon-dependent errors are compared with diffusion-free and latent-free architectural ablations, a persistence predictor, and linear extrapolation. Evaluation uses chronological partitions of a reproducible music-like waveform and reports mean squared prediction error at each forecast horizon, overall rollout error, and qualitative spectral changes. The primary hypothesis is that continuous-time coupling reduces error accumulation during evolving harmonic and transient events; a competing hypothesis is that integration error and spectral oversmoothing offset any benefit. The implemented rollout reinitializes the auxiliary latent variable between one-step calls, thereby isolating repeated-step forecast behaviour rather than persistent latent-memory effects. Accordingly, the experiment provides an ablation-oriented diagnostic study and does not establish superiority over persistent-state models or performance on unseen recordings. Results and comparative claims are deliberately deferred until notebook execution.

## 1. Introduction and research objective

Short-horizon forecasts are frequently evaluated under teacher forcing or independent one-step prediction. Such protocols can obscure compounding error when a state-space model is executed autonomously. For coupled spectral dynamics [1], a central question is whether local–global coupling reduces that compounding error or merely produces smoother one-step predictions.

**Research question (RQ2).** Does the proposed coupled neural PDE–ODE exhibit lower eight-step error accumulation than its architectural ablations and elementary forecasting controls when no intermediate ground-truth frames are provided?

**Scope.** The current implementation resets the global latent state during each `step` call. It is not a test of persistent context memory; that experiment belongs to the separate recording-level benchmark.

### 1.1. Unresolved problem: smooth dynamics versus discontinuous musical events

A classical ODE describes a continuously evolving hidden state, while music contains discrete events such as note attacks, percussion strikes, instrument changes and abrupt section boundaries. A smooth learned vector field may approximate these changes by adopting a very high local derivative, which can increase solver cost and destabilize forecasts. Eight-step autoregression exposes the onset of accumulated error but does not yet test genuine multi-phrase memory.

**Critical question CQ2 — hybrid event dynamics.** Can a model that combines continuous spectral evolution with learned event-triggered state jumps reproduce transients more faithfully and at lower compute cost than purely continuous PDE–ODE models?

**Research direction.** Extend the coupled equation with event surfaces and jump maps, \( (u,z)^+=\mathcal J_\eta((u,z)^-,e_t)\), while retaining the continuous flow between event times. Compare event-aware integration, continuous-only integration, and sequence baselines using exactly matched histories.

**Falsifiable criterion.** Event-aware dynamics should reduce onset timing error and transient spectral distortion on *annotated* recordings without degrading stable regions; solver evaluations per second must be reported. The present eight-step experiment, which resets its latent variable between steps, **does not implement jump dynamics** and cannot establish this claim. Onset supervision [5], Neural CDEs [6], and ODE-LSTMs [7] provide relevant prior art and competing designs.

**Critical question CQ3 — multi-timescale musical memory.** Can fast spectral dynamics and slower phrase-level structure be represented without gradual drift or loss of recurring motifs? An eight-frame horizon is too short to test this question. An appropriate study needs seconds-to-minutes forecasting with structural recurrence and section-level measurements, compared with hierarchical music models [8].

## 2. Proposed method

Let $u_k$ represent a normalized log-spectrum, and let $F_{\Delta t}$ denote the learned one-step numerical flow of the coupled system introduced in [1]. We investigate autonomous composition,

$$
\hat{u}_{k+h}=F_{\Delta t}^{\circ h}(u_k),\qquad h=1,\ldots,8,
$$

without substituting observed intermediate frames. Prediction risk is reported separately by horizon,

$$
\mathcal{E}_h=\frac{1}{NF}\sum_{i=1}^{N}\left\|\hat{u}^{(i)}_{k+h}-u^{(i)}_{k+h}\right\|_2^2.
$$

The coupled candidate contains convolutional reaction, nonnegative spectral diffusion, and a global latent ODE. The no-diffusion and no-latent variants control the principal architectural components. Persistence and linear extrapolation measure whether neural numerical integration adds value beyond trivial temporal assumptions.

**Design qualification.** Because $z$ is reset between calls, any difference identifies behaviour of repeated one-step coupled dynamics, *not* the contribution of latent state carried persistently across the complete horizon.

## 3. Expected outcomes and falsifiable hypotheses

**H1 (rollout stability).** On the controlled sequence, the coupled predictor exhibits a smaller increase in $\mathcal{E}_h$ over horizon than at least one ablation. **H0:** its horizon-averaged MSE is not smaller than the strongest control. **H2 (transient sensitivity).** Spectral peaks and onset-like changes produce disproportionately large error, potentially aggravated by diffusion.

No outcome is presumed. The predicted advantages must be evaluated against all control methods and accompanied by error-versus-horizon plots. This experiment is not a substitute for cross-recording tests.

## 4. Experimental procedure

**Step 1 — Observation generation.** Construct music-like waveforms or accept licensed audio and calculate log-spectral features. **Step 2 — Candidate training.** Fit complete and ablated models using the same one-step loss and chronological validation. **Step 3 — Free-running rollout.** Starting from withheld test frames, predict eight consecutive futures without teacher forcing. **Step 4 — Analysis.** Compare horizon-dependent errors with persistence and linear extrapolation; inspect error growth rather than only final-frame accuracy. The cells below implement this sequence.


### Explainability diagrams

The method graph emphasizes the continuous-time coupling; the evaluation graph distinguishes it from one-step and extrapolation controls. The decision diagram explains how a validation gate would prevent overfitted repairs in a future PCWSR extension.

### Interpretation of the methodological figures

**Figure 1 — Mechanistic architecture:** the highlighted node identifies the *proposed scientific mechanism*, not a generic preprocessing step. The incoming arrows specify evidence or state needed for the method; outgoing arrows specify the predictions or edits being evaluated.

**Figure 2 — Comparative evaluation:** proposed-method results are distinguished from baselines using a thick outline or hatching. A visually highlighted method is **not** an assumed winner. Its score must be measured on the same held-out target as the baselines.

**Figure 3 — Data partitioning and verification:** training diagnostics may inform candidate generation; validation decides acceptance; the test partition is reserved for final reporting. Failed model downloads, invalid outputs, and rejected repairs remain visible and are not replaced by fabricated scores.

These diagrams are executable Graphviz figures generated inside Colab; the paper-style explanations and executable experiments remain the primary deliverable.


In [ ]:
%pip -q install graphviz torchdiffeq soundfile scipy pandas matplotlib

In [ ]:
import os, math, time, random, json
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import torch
from torch import nn
from torchdiffeq import odeint
from scipy.signal import stft, resample_poly
from scipy.io import wavfile
SEED=211100343
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:',device,'torch:',torch.__version__)


In [ ]:
# Three executable research diagrams, saved inside the Colab runtime.
# The thicker proposed-method border is intentionally distinct from baseline methods.
from IPython.display import display, SVG
from graphviz import Digraph

def research_diagram(name, nodes, edges, proposed, caption):
    dot=Digraph(name,format='svg')
    dot.attr(rankdir='LR',fontname='Helvetica',labelloc='t',label=caption,nodesep='0.35',ranksep='0.45')
    for key,label in nodes:
        if key in proposed:
            dot.node(key,label,shape='box',style='rounded,bold',color='#B45309',penwidth='3.5',
                     fillcolor='#FFF7ED',fontcolor='#7C2D12')
        else:
            dot.node(key,label,shape='box',style='rounded',color='#64748B')
    for u,v in edges: dot.edge(u,v)
    try:
        svg=dot.pipe(format='svg').decode('utf-8')
        display(SVG(svg));Path(name+'.svg').write_text(svg)
    except Exception as exc:
        print('Graphviz rendering unavailable:',exc,'\n',dot.source)

research_diagram('v2_method',
    [('audio','Audio / spectrogram'),('diagnose','Diagnostics / local errors'),
     ('local','Local PDE-inspired operator'),('global','Persistent or short-horizon ODE'),
     ('proposer','PROPOSED: model / weight proposer'),('solve','Differentiable integration'),
     ('verify','Independent verifier')],
    [('audio','diagnose'),('diagnose','local'),('diagnose','global'),
     ('local','proposer'),('global','proposer'),('proposer','solve'),('solve','verify')],
    ['local','global','proposer','solve'], 'METHOD: '+'Error propagation across eight forecast steps')
research_diagram('v2_evaluation',
    [('data','Recordings / synthetic signals'),('split','Train → validation → test'),
     ('proposed','PROPOSED: Coupled rollout'),('base','Classical / neural baselines'),
     ('hf','3 Hugging Face models (where applicable)'),('gate','Validation / selection gate'),
     ('out','Held-out MSE, onsets, latency')],
    [('data','split'),('split','proposed'),('split','base'),('split','hf'),
     ('proposed','gate'),('base','gate'),('hf','gate'),('gate','out')],
    ['proposed'], 'EVALUATION: isolate the proposed method from benchmarks')
research_diagram('v2_decisions',
    [('frozen','Frozen candidate / fixed design'),('probe','Residual / sensitivity probe'),
     ('action','PROPOSED: targeted intervention'),('validation','Disjoint validation'),
     ('accepted','Accept only if improvement'),('rejected','Reject / record failure'),
     ('test','One-shot held-out evaluation')],
    [('frozen','probe'),('probe','action'),('action','validation'),
     ('validation','accepted'),('validation','rejected'),('accepted','test'),('rejected','test')],
    ['action'], 'VERIFICATION: no candidate accesses test labels during selection')

## Step 1 — Prepare a continuous spectral signal

In [ ]:
SAMPLE_RATE=8000
DURATION=5.0
USE_UPLOAD=False
def synth_audio(seed=3):
    rng=np.random.default_rng(seed)
    t=np.arange(int(SAMPLE_RATE*DURATION))/SAMPLE_RATE
    x=np.zeros_like(t)
    chords=[[220,277.18,329.63],[196,246.94,293.66],[174.61,220,261.63],[164.81,207.65,246.94]]
    for j,chord in enumerate(chords):
        a,b=j*1.25,(j+1)*1.25
        idx=(t>=a)&(t<b)
        tau=t[idx]-a
        envelope=np.minimum(1,tau*18)*np.exp(-0.35*tau)
        for f in chord:
            x[idx]+=envelope*np.sin(2*np.pi*f*tau+rng.uniform(-.2,.2))/len(chord)
    for beat in np.arange(0,DURATION,.5):
        ix=int(beat*SAMPLE_RATE)
        length=min(500,len(x)-ix)
        if length>0:
            pulse=rng.standard_normal(length)*np.exp(-np.arange(length)/90)
            x[ix:ix+length]+=.09*pulse
    return (x/(max(1e-6,np.max(np.abs(x))))).astype(np.float32)
if USE_UPLOAD:
    from google.colab import files
    upload=files.upload()
    fname=next(iter(upload))
    sr,audio=wavfile.read(fname)
    if audio.ndim>1: audio=audio.astype(np.float32).mean(axis=1)
    audio=audio.astype(np.float32)
    audio/=max(1e-6,np.max(np.abs(audio)))
    if sr!=SAMPLE_RATE:
        from math import gcd
        d=gcd(int(sr),SAMPLE_RATE)
        audio=resample_poly(audio,SAMPLE_RATE//d,int(sr)//d)
    audio=audio[:int(DURATION*SAMPLE_RATE)]
else:
    audio=synth_audio()
print('Samples:',len(audio),'duration:',round(len(audio)/SAMPLE_RATE,2),'s')
freqs,times,Z=stft(audio,fs=SAMPLE_RATE,nperseg=256,noverlap=192,boundary=None)
spec=np.log1p(10*np.abs(Z)).astype(np.float32)
# 64 frequency bands, using first 128 FFT bins in pairs
spec=spec[:128].reshape(64,2,-1).mean(axis=1)
print('Spectrogram:',spec.shape)
fig,ax=plt.subplots(figsize=(11,3))
ax.imshow(spec,aspect='auto',origin='lower',extent=[times[0],times[-1],0,64])
ax.set(xlabel='Time (s)',ylabel='Frequency band',title='Log magnitude spectrogram')
plt.show()


## Step 2 — Define the proposed coupled spectral dynamics

In [ ]:
BANDS=64
class CoupledDynamics(nn.Module):
    def __init__(self, mode='coupled', hidden=8):
        super().__init__()
        assert mode in ('coupled','no_diffusion','no_latent')
        self.mode=mode
        self.hidden=hidden
        self.diffusion_raw=nn.Parameter(torch.tensor(-2.0))
        self.local=nn.Sequential(nn.Conv1d(1,8,5,padding=2),nn.Tanh(),
                                 nn.Conv1d(8,1,3,padding=1))
        self.global_net=nn.Sequential(nn.Linear(hidden+1,16),nn.Tanh(),
                                      nn.Linear(16,hidden))
        self.inject=nn.Linear(hidden,1,bias=False)
    def forward(self,t,state):
        u,z=state
        # u: batch x bands, z: batch x hidden
        padded=torch.nn.functional.pad(u.unsqueeze(1),(1,1),mode='replicate').squeeze(1)
        lap=padded[:,:-2]-2*u+padded[:,2:]
        D=torch.nn.functional.softplus(self.diffusion_raw)
        if self.mode=='no_diffusion': D=D*0
        reaction=self.local(u.unsqueeze(1)).squeeze(1)
        if self.mode=='no_latent':
            dz=torch.zeros_like(z)
            coupling=torch.zeros_like(u)
        else:
            dz=self.global_net(torch.cat([z,u.mean(dim=-1,keepdim=True)],dim=-1))
            coupling=self.inject(z).expand_as(u)
        return D*lap+reaction+coupling,dz
    def step(self,u,dt):
        z=torch.zeros(u.shape[0],self.hidden,device=u.device,dtype=u.dtype)
        # Differentiable fixed-step integration of the coupled ODE
        t=torch.tensor([0.,float(dt)],device=u.device,dtype=u.dtype)
        traj=odeint(self,(u,z),t,method='rk4',options={'step_size':float(dt)/2})
        return traj[0][-1]


## Step 3 — Create a chronological pilot split and fit the one-step model

In [ ]:
X=spec.T
# normalize on first 60% of chronological frames only
N=len(X); n_train=int(.6*N); n_val=int(.8*N)
mu=X[:n_train].mean(axis=0,keepdims=True)
sd=X[:n_train].std(axis=0,keepdims=True)+1e-3
Xn=np.clip((X-mu)/sd,-5,5).astype(np.float32)
def pairs(start,end):
    return (torch.tensor(Xn[start:end-1],device=device),
            torch.tensor(Xn[start+1:end],device=device))
train_x,train_y=pairs(0,n_train)
val_x,val_y=pairs(n_train,n_val)
test_x,test_y=pairs(n_val,N)
DT=float(np.median(np.diff(times)))
print('train/val/test pairs:',len(train_x),len(val_x),len(test_x),'dt:',DT)


In [ ]:
import copy
EPOCHS=35
BATCH=32
results={}
predictions={}
for mode in ('coupled','no_diffusion','no_latent'):
    torch.manual_seed(SEED)
    model=CoupledDynamics(mode=mode).to(device)
    opt=torch.optim.Adam(model.parameters(),lr=2e-3)
    best=float('inf'); best_state=None
    started=time.perf_counter()
    for epoch in range(EPOCHS):
        model.train()
        for ix in torch.randperm(len(train_x),device=device).split(BATCH):
            opt.zero_grad()
            pred=model.step(train_x[ix],DT)
            loss=(pred-train_y[ix]).square().mean()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(),1.)
            opt.step()
        model.eval()
        with torch.no_grad():
            val=float((model.step(val_x,DT)-val_y).square().mean())
        if val<best:
            best=val
            best_state=copy.deepcopy(model.state_dict())
    model.load_state_dict(best_state)
    with torch.no_grad():
        pred=model.step(test_x,DT)
        mse=float((pred-test_y).square().mean())
        mae=float((pred-test_y).abs().mean())
        predictions[mode]=pred.detach().cpu().numpy()
    results[mode]={'test_mse':mse,'test_mae':mae,'validation_mse':best,
                   'training_seconds':time.perf_counter()-started,
                   'diffusion':float(torch.nn.functional.softplus(model.diffusion_raw))}
    print(mode,results[mode])
results['persistence']={'test_mse':float((test_x-test_y).square().mean()),
                        'test_mae':float((test_x-test_y).abs().mean())}
print('persistence',results['persistence'])


## Step 4 — Construct autoregressive multi-step evaluation

We use the model's previous prediction as the next input without teacher forcing. The target is an eight-step sequence. **Caveat:** in this pilot, the model's latent state is reset by `step`; the cross-recording V2 notebook implements persistent joint integration.

In [ ]:
# Keep trained models: rerun training cell with this small modification if needed.
# The original training loop did not retain models. This cell re-trains all variants
# reproducibly, then evaluates a strictly autoregressive 8-step rollout.
import copy
HORIZON=8
ROLLOUT_EPOCHS=35
ROLLOUT_BATCH=32
def fit_rollout_model(mode):
    torch.manual_seed(SEED)
    model=CoupledDynamics(mode=mode).to(device)
    optimizer=torch.optim.Adam(model.parameters(),lr=2e-3)
    best_val=float('inf'); best_state=None
    for epoch in range(ROLLOUT_EPOCHS):
        model.train()
        for ix in torch.randperm(len(train_x),device=device).split(ROLLOUT_BATCH):
            optimizer.zero_grad()
            pred=model.step(train_x[ix],DT)
            loss=(pred-train_y[ix]).square().mean()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(),1.)
            optimizer.step()
        model.eval()
        with torch.no_grad():
            score=float((model.step(val_x,DT)-val_y).square().mean())
        if score<best_val:
            best_val=score
            best_state=copy.deepcopy(model.state_dict())
    model.load_state_dict(best_state)
    model.eval()
    return model,best_val

rollout_models={}
for mode in ('coupled','no_diffusion','no_latent'):
    model,val=fit_rollout_model(mode)
    rollout_models[mode]=model
    print('Trained',mode,'validation MSE',round(val,6))


In [ ]:
# Construct forecast windows entirely within the held-out chronological test block.
# The test target is a matrix [number of windows, horizon, frequency bands].
test_block=torch.tensor(Xn[n_val:],dtype=torch.float32,device=device)
if len(test_block)<HORIZON+2:
    raise ValueError('Test block too short; increase audio duration or reduce HORIZON.')
starts=torch.arange(len(test_block)-HORIZON,device=device)
initial=test_block[starts]
targets=torch.stack([test_block[starts+h] for h in range(1,HORIZON+1)],dim=1)
rollout_predictions={}
with torch.no_grad():
    for name,model in rollout_models.items():
        state=initial
        frames=[]
        for _ in range(HORIZON):
            state=model.step(state,DT)
            frames.append(state)
        rollout_predictions[name]=torch.stack(frames,dim=1)
    rollout_predictions['persistence']=initial[:,None,:].expand_as(targets)
    # Linear extrapolation uses only two already-observed frames.
    valid_starts=starts[1:]
    prev=test_block[valid_starts-1]
    curr=test_block[valid_starts]
    linear=torch.stack([curr+h*(curr-prev) for h in range(1,HORIZON+1)],dim=1)
    rollout_predictions['linear_extrapolation']=torch.cat(
        [rollout_predictions['persistence'][:1],linear],dim=0)

rollout_mse={}
for name,pred in rollout_predictions.items():
    per_horizon=((pred-targets)**2).mean(dim=(0,2)).cpu().numpy()
    rollout_mse[name]=per_horizon
    print(f'{name:22s} mean={per_horizon.mean():.6f} final={per_horizon[-1]:.6f}')


## 5. Experimental evaluation — Error accumulation in proposed versus baseline predictions

We report MSE separately by horizon. The proposed model is highlighted in the V2 summary and is deemed stronger only when the held-out mean rollout MSE is better than persistence and extrapolation.

In [ ]:
fig,ax=plt.subplots(1,2,figsize=(14,4.5))
for name,err in rollout_mse.items():
    ax[0].plot(np.arange(1,HORIZON+1),err,marker='o',label=name)
ax[0].set(xlabel='Forecast horizon (frames)',ylabel='Held-out spectral MSE ↓',
          title='Autoregressive forecast degradation')
ax[0].legend(fontsize=8)
names=list(rollout_mse)
means=[float(rollout_mse[k].mean()) for k in names]
ax[1].bar(names,means)
ax[1].tick_params(axis='x',rotation=22)
ax[1].set(ylabel='Mean 1–8 step MSE ↓',title='Difficult-task comparison')
fig.tight_layout()
fig.savefig('music_multistep_comparison.png',dpi=160)
plt.show()

rollout_table=pd.DataFrame({'model':names,
                            'mean_rollout_mse':means,
                            'final_horizon_mse':[float(rollout_mse[k][-1]) for k in names]})
rollout_table.to_csv('music_multistep_metrics.csv',index=False)
best_name=rollout_table.sort_values('mean_rollout_mse').iloc[0]['model']
print('Lowest mean held-out rollout error:',best_name)
print(rollout_table.sort_values('mean_rollout_mse').to_string(index=False))
print('Saved music_multistep_metrics.csv and music_multistep_comparison.png')


In [ ]:
complex_df=pd.DataFrame({'model':list(rollout_mse),'mean_rollout_mse':[float(v.mean()) for v in rollout_mse.values()]})
print(complex_df.sort_values('mean_rollout_mse').to_string(index=False))
print('Proposed is best on this pilot:',complex_df.loc[complex_df.model=='coupled','mean_rollout_mse'].iloc[0] == complex_df.mean_rollout_mse.min())

In [ ]:
# A publication-oriented plot: the evaluated proposal is visually distinct.
fig,ax=plt.subplots(figsize=(11,4.5))
plot_data=complex_df
plot_names=plot_data['model'].astype(str).tolist()
plot_values=plot_data['mean_rollout_mse'].astype(float).tolist()
bars=ax.bar(plot_names,plot_values,alpha=.64)
for label,bar in zip(plot_names,bars):
    if label=='coupled':
        bar.set_edgecolor('black');bar.set_linewidth(3);bar.set_hatch('///')
        ax.annotate('PROPOSED METHOD', (bar.get_x()+bar.get_width()/2,bar.get_height()),
                    xytext=(0,12),textcoords='offset points',ha='center',fontsize=8,weight='bold')
ax.set(ylabel='mean_rollout_mse (lower is better)',title='Eight-step forecasting: proposed coupled rollout')
ax.tick_params(axis='x',rotation=32)
fig.tight_layout();fig.savefig('v2_complex_proposed_comparison.png',dpi=160);plt.show()

## 6. Interpretation of numerical results

After executing the notebook, use the reported held-out metrics to determine whether the proposed coupled dynamics outperform the ablations and simple baselines. A win on one synthetic signal is **not** evidence of real-music generalization. If the proposed model loses, preserve that negative result and investigate diffusion strength, temporal context, and baseline capacity.

**Future research:** repeat across independent recordings and random seeds, compare against three pretrained Hugging Face forecasting checkpoints in the recording-level V2 notebook, and evaluate persistent state and targeted weight-space repair separately.


## Research agenda: evidentiary standards

The research questions above are intentionally harder than the experiments currently executed by this notebook. **A hypothesis is not a measured result**, and an unimplemented extension is not a contribution. The diagrams identify the proposed scientific mechanism; they must not visually encode superiority that the numerical evidence has not established.

A claim of superiority requires a preregistered outcome, independent train/validation/test groups, repeated random seeds, uncertainty estimates, robust baselines, and numerical/compute fairness. When comparing pretrained Hugging Face models, distinguish time-series forecasting checkpoints from instruction models proposing weight edits. Unavailable checkpoints and rejected repairs remain explicitly missing/rejected, never assigned fabricated performance.

The highest-impact unresolved questions are (i) separating smooth dynamics from abrupt musical events, (ii) identifying transferable local/global representations, (iii) preserving harmonic and phase detail, (iv) maintaining long-range structure, and (v) performing non-regressive, localized neural weight-space repair under distribution shift. The present notebook provides an initial component of that program and specifies which claims remain future work.


## 7. Discussion and limitations

**Discussion.** The experiment measures short autoregressive stability, not learned music semantics. The synthetic waveform is limited, and the model resets the latent state between forecast steps. Any observed improvement cannot be attributed to retained global context. Similarly, a low one-step loss is not evidence of robust long-horizon dynamics.

## 8. Conclusion and future research

**Conclusion.** The coupled PDE–ODE hypothesis will be considered supported only within the observed experimental setting if autonomous rollout error improves over the relevant controls. Both adverse error accumulation and failure to beat persistence are informative outcomes that must be retained.

**Future research.** Jointly integrate $(u,z)$ over the full horizon, include increasingly long forecasts and irregularly sampled observations, evaluate recording-disjoint music data, and compare against established sequence forecasters and pretrained time-series models. Test whether spectral diffusion destroys short-duration transients and whether the computational cost is justified.

## References

[1] M. Habiba and B. A. Pearlmutter, *Continuous Convolutional Neural Networks: Coupled Neural PDE and ODE* (2021). https://arxiv.org/abs/2111.00343

[2] R. T. Q. Chen et al., *Neural Ordinary Differential Equations*, NeurIPS (2018). https://arxiv.org/abs/1806.07366

[3] R. T. Q. Chen, `torchdiffeq`. https://github.com/rtqichen/torchdiffeq

[4] M. Habiba and B. A. Pearlmutter, *Neural Ordinary Differential Equation based Recurrent Neural Network Model* (2020). https://arxiv.org/abs/2005.09807

**Reproducibility.** The pilot uses a controlled seed and chronologically withheld test windows. Repeated trials and recording-disjoint evaluation are required before publication-level claims.
[5] C. Hawthorne et al., *Onsets and Frames: Dual-Objective Piano Transcription*, ISMIR (2018). https://arxiv.org/abs/1710.11153

[6] P. Kidger et al., *Neural Controlled Differential Equations for Irregular Time Series*, NeurIPS (2020). https://papers.neurips.cc/paper/2020/hash/4a5876b450b45371f6cfe5047ac8cd45-Abstract.html

[7] M. Lechner and R. Hasani, *Learning Long-Term Dependencies in Irregularly-Sampled Time Series* (2020). https://arxiv.org/abs/2006.04418

[8] X. Wang, H. Wang, and W. Cai, *MusicWeaver: Coherent Long-Range and Editable Music Generation from a Beat-Aligned Structural Plan* (2025). https://arxiv.org/abs/2509.21714
